# Data prep — every case, with examples
Each cell is one case: run it to see before → after, then copy the line you need into Q1 / Q2 cell 1.
Toy table `t` for the small examples, `df` = titanic for the real ones.

| Missing value situation | Use |
|---|---|
| few rows missing | `dropna(subset=[...])` |
| column mostly empty (> 40 %) | `drop(columns=[...])` |
| number, normal | fill **mean** |
| number, skewed / money / outliers | fill **median** |
| missing means "none" (count, bonus) | fill **0** |
| text / category | fill **mode** or `"Unknown"` |
| depends on a group | fill **group median** |
| y (target) missing | **drop the row**, never fill |

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../titanic.csv")

def toy():
    return pd.DataFrame({
        "age":    [22, np.nan, 35, np.nan, 58, 40],
        "city":   ["BKK", None, "CNX", "BKK", None, "BKK"],
        "salary": ["1,200", "?", "3,400", None, "5,000", "2,100"],
        "bonus":  [100, np.nan, np.nan, 50, 0, 20],
        "grade":  ["A", "C", "B", "B", "A", "C"],
        "date":   ["01/03/2024", "15/03/2024", "20/04/2024", "02/05/2024", "30/06/2024", "07/07/2024"],
    })
t = toy()
t

,age,city,salary,bonus,grade,date
0,22.0,BKK,"1,200",100.0,A,01/03/2024
1,NaN,NaN,?,NaN,C,15/03/2024
2,35.0,CNX,"3,400",NaN,B,20/04/2024
3,NaN,BKK,NaN,50.0,B,02/05/2024
4,58.0,NaN,"5,000",0.0,A,30/06/2024
5,40.0,BKK,"2,100",20.0,C,07/07/2024


## 1. Look at the data

In [2]:
print(df.shape)                    # rows, columns
print(df.columns.tolist())         # exact column names (copy them!)
df.info()                          # types + non-null count

(891, 12)
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


In [3]:
print(df.isnull().sum())                       # missing per column
print((df.isnull().mean() * 100).round(1))     # missing %

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age            19.9
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Cabin          77.1
Embarked        0.2
dtype: float64


In [4]:
print(df.describe())                           # min / max / mean -> spot strange values
print(df["Embarked"].value_counts())           # count each category
print(df.nunique())                            # unique values per column (≈ rows -> id column, drop it)
print("duplicates:", df.duplicated().sum())

       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.836071   14.526497    1.102743   
min       1.000000    0.000000    1.000000    0.420000    0.000000   
25%     223.500000    0.000000    2.000000   20.125000    0.000000   
50%     446.000000    0.000000    3.000000   28.000000    0.000000   
75%     668.500000    1.000000    3.000000   38.000000    1.000000   
max     891.000000    1.000000    3.000000   80.000000    8.000000   

            Parch        Fare  
count  891.000000  891.000000  
mean     0.381594   32.204208  
std      0.806057   49.693429  
min      0.000000    0.000000  
25%      0.000000    7.910400  
50%      0.000000   14.454200  
75%      0.000000   31.000000  
max      6.000000  512.329200  
Embarked
S    644
C    168
Q     77
Name: count, dtype: int64
PassengerId    891


## 2. Drop rows

In [5]:
t = toy()
print(t.dropna().shape)                        # drop row if ANY column is missing
print(t.dropna(subset=["age"]).shape)          # drop row only if age is missing
print(t.dropna(how="all").shape)               # drop row only if ALL columns are missing
print(t.dropna(thresh=5).shape)                # keep rows with at least 5 non-missing values
print(df.dropna().shape, "<- titanic: dropna() on everything kills most rows (Cabin)!")

(2, 6)
(4, 6)
(6, 6)
(4, 6)
(183, 12) <- titanic: dropna() on everything kills most rows (Cabin)!


## 3. Drop columns

In [6]:
d = df.drop(columns=["Cabin"])                                  # one column
d = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"]) # several
d = df.loc[:, df.isnull().mean() < 0.4]                         # every column with < 40 % missing
print(d.columns.tolist())
d = df[["Pclass", "Sex", "Age", "Fare"]]                        # or just KEEP the ones you want

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']


## 4. Fill missing — numbers

In [7]:
t = toy()
t["age_0"]      = t["age"].fillna(0)                   # fill 0
t["age_mean"]   = t["age"].fillna(t["age"].mean())     # fill mean
t["age_median"] = t["age"].fillna(t["age"].median())   # fill median
t["age_99"]     = t["age"].fillna(99)                  # fill any constant
t["age_ffill"]  = t["age"].ffill()                     # copy the value from the row above
t["age_bfill"]  = t["age"].bfill()                     # copy the value from the row below
t["age_interp"] = t["age"].interpolate()               # straight line between neighbours
t.filter(like="age")

,age,age_0,age_mean,age_median,age_99,age_ffill,age_bfill,age_interp
0,22.0,22.0,22.00,22.0,22.0,22.0,22.0,22.0
1,NaN,0.0,38.75,37.5,99.0,22.0,35.0,28.5
2,35.0,35.0,35.00,35.0,35.0,35.0,35.0,35.0
3,NaN,0.0,38.75,37.5,99.0,35.0,58.0,46.5
4,58.0,58.0,58.00,58.0,58.0,58.0,58.0,58.0
5,40.0,40.0,40.00,40.0,40.0,40.0,40.0,40.0


In [8]:
# fill ALL number columns at once
d = df.copy()
num = d.select_dtypes(include="number").columns
d[num] = d[num].fillna(d[num].median())       # or .mean() or 0
print(d[num].isnull().sum().sum(), "missing numbers left")

# a different fill for each column
t = toy()
t = t.fillna({"age": t["age"].median(), "bonus": 0, "city": "Unknown"})
t

0 missing numbers left


,age,city,salary,bonus,grade,date
0,22.0,BKK,"1,200",100.0,A,01/03/2024
1,37.5,Unknown,?,0.0,C,15/03/2024
2,35.0,CNX,"3,400",0.0,B,20/04/2024
3,37.5,BKK,NaN,50.0,B,02/05/2024
4,58.0,Unknown,"5,000",0.0,A,30/06/2024
5,40.0,BKK,"2,100",20.0,C,07/07/2024


In [9]:
# group fill: missing Age = median Age of people with the same Pclass and Sex
d = df.copy()
d["Age"] = d["Age"].fillna(d.groupby(["Pclass", "Sex"])["Age"].transform("median"))
print(d["Age"].isnull().sum(), "missing Age left")

# keep the fact that it was missing (sometimes helps the model)
d = df.copy()
d["Age_missing"] = d["Age"].isnull().astype(int)
d["Age"] = d["Age"].fillna(d["Age"].median())
d[["Age", "Age_missing"]].head(7)

0 missing Age left


,Age,Age_missing
0,22.0,0
1,38.0,0
2,26.0,0
3,35.0,0
4,35.0,0
5,28.0,1
6,54.0,0


## 5. Fill missing — text

In [10]:
t = toy()
t["city_mode"]    = t["city"].fillna(t["city"].mode()[0])   # most common value
t["city_unknown"] = t["city"].fillna("Unknown")             # new category
t[["city", "city_mode", "city_unknown"]]

,city,city_mode,city_unknown
0,BKK,BKK,BKK
1,NaN,BKK,Unknown
2,CNX,CNX,CNX
3,BKK,BKK,BKK
4,NaN,BKK,Unknown
5,BKK,BKK,BKK


In [11]:
# ALL text columns at once
d = df.copy()
txt = d.select_dtypes(exclude="number").columns
for c in txt:
    d[c] = d[c].fillna(d[c].mode()[0])
print(d.isnull().sum().sum(), "missing left")

177 missing left


**pandas 3 trap:** `df["Age"].fillna(0, inplace=True)` now does **nothing**.
Always assign the result back: `df["Age"] = df["Age"].fillna(0)`

## 6. Weird values that mean missing (`?`, `-999`, blank)

In [12]:
t = toy()
t = t.replace({"?": np.nan, "": np.nan, -999: np.nan})
t["salary"]

0    1,200
1      NaN
2    3,400
3      NaN
4    5,000
5    2,100
Name: salary, dtype: str

## 7. Text that should be numbers

In [13]:
t = toy()
t["salary"] = t["salary"].replace("?", np.nan)
t["salary"] = t["salary"].str.replace(",", "")                  # "1,200" -> "1200"
t["salary"] = pd.to_numeric(t["salary"], errors="coerce")       # bad text -> NaN
t["salary"] = t["salary"].fillna(t["salary"].median())
print(t["salary"].tolist())

s = pd.Series(["$50", "12 km", "7.5%", " 30 "])
print(pd.to_numeric(s.str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce").tolist())   # keep only digits

# clean text categories
s = pd.Series([" BKK", "bkk ", "Bkk"])
print(s.str.strip().str.lower().tolist())

[1200.0, 2750.0, 3400.0, 2750.0, 5000.0, 2100.0]
[50.0, 12.0, 7.5, 30.0]
['bkk', 'bkk', 'bkk']


## 8. Categories → numbers

In [14]:
d = df.copy()
# 2 categories -> 0 / 1
d["Sex"] = d["Sex"].map({"male": 1, "female": 0})
# or: d["Sex"] = (d["Sex"] == "male").astype(int)

# ordered categories -> keep the order
t = toy()
t["grade_num"] = t["grade"].map({"C": 0, "B": 1, "A": 2})

# many categories, no order -> one-hot (0/1 column per category)
d = pd.get_dummies(d, columns=["Embarked"], drop_first=True, dtype=int)
print(d.filter(like="Embarked").head())

# quick numbers for every category (fine for tree models)
t["city_code"] = t["city"].astype("category").cat.codes      # NaN -> -1
t[["grade", "grade_num", "city", "city_code"]]

   Embarked_Q  Embarked_S
0           0           1
1           0           0
2           0           1
3           0           1
4           0           1


,grade,grade_num,city,city_code
0,A,2,BKK,0
1,C,0,NaN,-1
2,B,1,CNX,1
3,B,1,BKK,0
4,A,2,NaN,-1
5,C,0,BKK,0


In [15]:
# rare categories -> "Other"  (e.g. Title from Name)
d = df.copy()
d["Title"] = d["Name"].str.extract(r",\s*([^\.]+)\.")[0]
counts = d["Title"].value_counts()
d["Title"] = d["Title"].where(d["Title"].map(counts) >= 10, "Other")
d["Title"].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64

## 9. Dates

In [16]:
t = toy()
t["date"] = pd.to_datetime(t["date"], dayfirst=True)      # 15/03/2024 = day first
t["year"] = t["date"].dt.year
t["month"] = t["date"].dt.month
t["day"] = t["date"].dt.day
t["weekday"] = t["date"].dt.dayofweek                     # 0 = Monday
t["weekend"] = (t["weekday"] >= 5).astype(int)
t = t.drop(columns=["date"])                              # models can't use the raw date
t[["year", "month", "day", "weekday", "weekend"]]

,year,month,day,weekday,weekend
0,2024,3,1,4,0
1,2024,3,15,4,0
2,2024,4,20,5,1
3,2024,5,2,3,0
4,2024,6,30,6,1
5,2024,7,7,6,1


## 10. Outliers

In [17]:
d = df.copy()
q1, q3 = d["Fare"].quantile([0.25, 0.75])
iqr = q3 - q1
print("rows:", len(d), "-> after IQR removal:", len(d[(d["Fare"] >= q1 - 1.5 * iqr) & (d["Fare"] <= q3 + 1.5 * iqr)]))

lo, hi = d["Age"].quantile([0.01, 0.99])
d["Age"] = d["Age"].clip(lo, hi)              # cap a FEATURE (safer than deleting)
d = d[d["Fare"] >= 0]                          # remove impossible values
# careful: do not delete outliers of the TARGET in the test set -> the teacher scores on it

rows: 891 -> after IQR removal: 775


## 11. New columns (free score boost)

In [18]:
d = df.copy()
d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
d["IsAlone"] = (d["FamilySize"] == 1).astype(int)
d["AgeGroup"] = pd.cut(d["Age"], bins=[0, 12, 18, 60, 100], labels=[0, 1, 2, 3])
d["FarePerPerson"] = d["Fare"] / d["FamilySize"]
d[["SibSp", "Parch", "FamilySize", "IsAlone", "Age", "AgeGroup"]].head()

,SibSp,Parch,FamilySize,IsAlone,Age,AgeGroup
0,1,0,2,0,22.0,2
1,1,0,2,0,38.0,2
2,0,0,1,1,26.0,2
3,1,0,2,0,35.0,2
4,0,0,1,1,35.0,2


## 12. Select / filter / rename / sort

In [19]:
d = df[["Pclass", "Sex", "Age", "Fare"]]                 # keep columns
d = df[df["Age"] > 18]                                    # filter rows
d = df[(df["Sex"] == "female") & (df["Pclass"] == 1)]     # 2 conditions: & and, | or
d = df[df["Embarked"].isin(["S", "C"])]
d = df.rename(columns={"Pclass": "Class"})
d = df.sort_values("Fare", ascending=False)
df.columns = df.columns.str.strip()                       # remove hidden spaces in column names
print(df.select_dtypes(include="number").columns.tolist())

['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']


## 13. Which features matter? (pick `cols`)

In [20]:
d = df.copy()
d["Sex"] = d["Sex"].map({"male": 1, "female": 0})
print(d.select_dtypes("number").corr()["Fare"].sort_values(key=abs, ascending=False))   # vs regression target
print(d.select_dtypes("number").corr()["Survived"].sort_values(key=abs, ascending=False))

Fare           1.000000
Pclass        -0.549500
Survived       0.257307
Parch          0.216225
Sex           -0.182333
SibSp          0.159651
Age            0.096067
PassengerId    0.012658
Name: Fare, dtype: float64
Survived       1.000000
Sex           -0.543351
Pclass        -0.338481
Fare           0.257307
Parch          0.081629
Age           -0.077221
SibSp         -0.035322
PassengerId   -0.005007
Name: Survived, dtype: float64


## 14. Make a class from a number (if the teacher has no class column)

In [21]:
d = df.copy()
d["expensive"] = (d["Fare"] > 30).astype(int)                                     # 2 classes
d["fare_level"] = pd.cut(d["Fare"], bins=[-1, 10, 50, 1000], labels=[0, 1, 2])     # 3 classes by ranges
d["fare_q"] = pd.qcut(d["Fare"], q=3, labels=[0, 1, 2])                            # 3 equal-size groups
d[["Fare", "expensive", "fare_level", "fare_q"]].head()

,Fare,expensive,fare_level,fare_q
0,7.2500,0,0,0
1,71.2833,1,2,2
2,7.9250,0,0,0
3,53.1000,1,2,2
4,8.0500,0,0,0


## 15. Scaling (KNN / Logistic / gradient descent like it)

In [22]:
x = df[["Age", "Fare"]].fillna(0).values
x_std = (x - x.mean(axis=0)) / x.std(axis=0)                           # standardise (scratch)
x_mm = (x - x.min(axis=0)) / (x.max(axis=0) - x.min(axis=0))           # 0..1 (scratch)
print(x_std[:3].round(2), "\n", x_mm[:3].round(2))
# sklearn: make_pipeline(StandardScaler(), KNeighborsClassifier())  -> scaler learns from train only

[[-0.1  -0.5 ]
 [ 0.81  0.79]
 [ 0.13 -0.49]] 
 [[0.28 0.01]
 [0.48 0.14]
 [0.32 0.02]]


## 16. Split train / test

In [23]:
from sklearn.model_selection import train_test_split
d = df.dropna(subset=["Age"])
x, y = d[["Pclass", "Age"]], d["Survived"]

# numpy way (Q1, from scratch)
d2 = d.sample(frac=1, random_state=42)
n = int(len(d2) * 0.8)
train, test = d2[:n], d2[n:]

# sklearn way (Q2); stratify keeps the same class % in train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(len(train), len(test), "|", len(x_train), len(x_test))

571 143 | 571 143


## 17. Full titanic recipe (copy this whole cell)

In [24]:
d = pd.read_csv("../titanic.csv")
d = d.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
d["Sex"] = d["Sex"].map({"male": 1, "female": 0})
d["Embarked"] = d["Embarked"].fillna(d["Embarked"].mode()[0])
d = pd.get_dummies(d, columns=["Embarked"], drop_first=True, dtype=int)
d["Age"] = d["Age"].fillna(d["Age"].median())
d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
print(d.isnull().sum().sum(), "missing |", d.shape)
d.head()

0 missing | (891, 10)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S,FamilySize
0,0,3,1,22.0,1,0,7.2500,0,1,2
1,1,1,0,38.0,1,0,71.2833,0,0,2
2,1,3,0,26.0,0,0,7.9250,0,1,1
3,1,1,0,35.0,1,0,53.1000,0,1,2
4,0,3,1,35.0,0,0,8.0500,0,1,1


## Errors → fix
| Error | Fix |
|---|---|
| `could not convert string to float` | a text column is still in x → `map` / `get_dummies` / drop it |
| `Input X contains NaN` | fill or drop missing first |
| `KeyError: 'Age'` | wrong name → `print(df.columns.tolist())`, `df.columns = df.columns.str.strip()` |
| everything in 1 column | `pd.read_csv(path, sep=";")` |
| Thai text broken | `pd.read_csv(path, encoding="utf-8-sig")` or `encoding="cp874"` |
| file not found | `pd.read_csv(r"C:\full\path\file.csv")` |
| `Singular matrix` | use `np.linalg.pinv` (already used) |
| `Target is multiclass` | `f1_score(..., average="macro")` |
| `fillna(inplace=True)` did nothing | `df["c"] = df["c"].fillna(...)` |
| `SettingWithCopyWarning` | add `.copy()` after filtering |